# Pipeline corrigé : train / validation / test
### Scoring de risque de crédit — Consolidation S6 à S9

**Pourquoi ce notebook existe.** En S6-S9, on a comparé des modèles, réglé des hyperparamètres et choisi un seuil de décision en regardant à chaque fois le score sur le même jeu de test. Résultat : le test a indirectement influencé plusieurs décisions, ce qui biaise l'évaluation finale de façon optimiste.

**La correction : un découpage à trois.**
- **Train (60%)** : entraîner les modèles.
- **Validation (20%)** : comparer les modèles, régler les hyperparamètres, choisir le seuil — toutes les décisions itératives.
- **Test (20%)** : une seule évaluation finale, jamais consultée avant la toute dernière étape.

On refait donc, dans l'ordre, tout le travail de S6 à S9 avec cette discipline stricte.

## Étape 0 — Préparer les données (S4-S5)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Loan_default.csv")

mois_emploi_maximum_plausible = (df["Age"] - 16) * 12
df["AgeEmploymentIncoherent"] = (df["MonthsEmployed"] > mois_emploi_maximum_plausible).astype(int)

for colonne in ["HasMortgage", "HasDependents", "HasCoSigner"]:
    df[colonne] = df[colonne].map({"Yes": 1, "No": 0})
df["Education_encoded"] = df["Education"].map({"High School": 0, "Bachelor's": 1, "Master's": 2, "PhD": 3})
df = pd.get_dummies(df, columns=["EmploymentType", "MaritalStatus", "LoanPurpose"],
                     prefix=["EmploymentType", "MaritalStatus", "LoanPurpose"])
colonnes_one_hot = [col for col in df.columns if col.startswith(("EmploymentType_", "MaritalStatus_", "LoanPurpose_"))]
for colonne in colonnes_one_hot:
    df[colonne] = df[colonne].astype(int)
df["LoanToIncomeRatio"] = df["LoanAmount"] / df["Income"]
df = df.drop(columns=["LoanID", "Education"])

X = df.drop(columns=["Default"])
y = df["Default"]
print("Donnees pretes :", X.shape)

Donnees pretes : (255347, 27)


## Étape 1 — Le découpage à trois, fait une seule fois

On procède en deux temps : d'abord séparer le test (20%) du reste, puis séparer ce "reste" en train (60% du total) et validation (20% du total). La stratification est appliquée aux deux étapes, pour préserver le déséquilibre 88%/12% dans les trois jeux.

In [2]:
from sklearn.model_selection import train_test_split

# Etape 1a : separer le test (20%) - on n'y touchera plus avant la toute fin
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Etape 1b : separer le reste en train (75% de X_temp = 60% du total) et validation (25% de X_temp = 20% du total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

print(f"Train      : {X_train.shape[0]:>7} lignes ({X_train.shape[0]/len(X)*100:.0f} %)")
print(f"Validation : {X_val.shape[0]:>7} lignes ({X_val.shape[0]/len(X)*100:.0f} %)")
print(f"Test       : {X_test.shape[0]:>7} lignes ({X_test.shape[0]/len(X)*100:.0f} %)")
print()
print("Proportion de defauts par jeu (doit rester ~11.6% partout) :")
print(f"Train      : {y_train.mean():.3f}")
print(f"Validation : {y_val.mean():.3f}")
print(f"Test       : {y_test.mean():.3f}")

Train      :  153207 lignes (60 %)
Validation :   51070 lignes (20 %)
Test       :   51070 lignes (20 %)

Proportion de defauts par jeu (doit rester ~11.6% partout) :
Train      : 0.116
Validation : 0.116
Test       : 0.116


## Étape 2 — Standardisation

Comme en S6 : on apprend la standardisation uniquement sur le train, puis on l'applique au validation ET au test avec les mêmes paramètres.

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Standardisation appliquee.")

Standardisation appliquee.


---
# Étape 3 — Comparer les modèles sur la validation (pas le test)

Reproduction de S6-S7, mais évaluée sur `validation` au lieu de `test`.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score

# Regression logistique (ponderee) - sur donnees standardisees
modele_logistique = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
modele_logistique.fit(X_train_scaled, y_train)

# Random Forest - sur donnees non standardisees (inutile pour les arbres)
modele_rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
modele_rf.fit(X_train, y_train)

# Gradient Boosting - sur donnees non standardisees
modele_gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42)
modele_gb.fit(X_train, y_train)

print("Trois modeles entraines sur le train.")

Trois modeles entraines sur le train.


In [5]:
comparaison_validation = pd.DataFrame({
    "Modele": ["Regression logistique", "Random Forest", "Gradient Boosting"],
    "AUC-ROC (validation)": [
        roc_auc_score(y_val, modele_logistique.predict_proba(X_val_scaled)[:, 1]),
        roc_auc_score(y_val, modele_rf.predict_proba(X_val)[:, 1]),
        roc_auc_score(y_val, modele_gb.predict_proba(X_val)[:, 1]),
    ],
    "Recall Defaut (validation)": [
        recall_score(y_val, modele_logistique.predict(X_val_scaled)),
        recall_score(y_val, modele_rf.predict(X_val)),
        recall_score(y_val, modele_gb.predict(X_val)),
    ],
})
comparaison_validation.round(3)

,Modele,AUC-ROC (validation),Recall Defaut (validation)
0,Regression logistique,0.755,0.691
1,Random Forest,0.747,0.591
2,Gradient Boosting,0.753,0.072


**Décision attendue, cohérente avec S7 :** les trois modèles restent proches en AUC ; on retient la régression logistique, la plus simple et la plus interprétable, à performance égale. Cette comparaison a utilisé `validation`, jamais `test` — le test reste donc intact.

---
# Étape 4 — Régler les hyperparamètres avec la validation

`GridSearchCV` utilise sa propre validation croisée *interne au train* pour chercher le meilleur C — c'est une pratique acceptée, car cette validation croisée reste entièrement à l'intérieur du train, sans toucher ni validation ni test. On confirme ensuite le choix sur le jeu de validation, séparément.

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

decoupage_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

recherche = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    param_grid={"C": [0.001, 0.01, 0.1, 1, 10, 100]},
    scoring="roc_auc",
    cv=decoupage_cv,
    n_jobs=-1
)
recherche.fit(X_train_scaled, y_train)

modele_final = recherche.best_estimator_

print("Meilleur C (trouve sur le train uniquement) :", recherche.best_params_)
print(f"AUC sur validation avec ce modele : {roc_auc_score(y_val, modele_final.predict_proba(X_val_scaled)[:, 1]):.3f}")

Meilleur C (trouve sur le train uniquement) : {'C': 100}
AUC sur validation avec ce modele : 0.755


---
# Étape 5 — Choisir le seuil de décision sur la validation

Reproduction de S8, mais la courbe precision/recall est calculée sur `validation`, jamais sur `test`.

In [7]:
from sklearn.metrics import precision_recall_curve

y_proba_val = modele_final.predict_proba(X_val_scaled)[:, 1]
precisions, recalls, seuils = precision_recall_curve(y_val, y_proba_val)

# Rappel de la lecon de S8 : ne pas optimiser F1 aveuglement, le recall est la priorite metier.
# On garde ici le seuil 0.5 (deja combine a class_weight='balanced'), comme decide en S8,
# mais on verifie sa performance sur la validation pour confirmer ce choix avant de toucher au test.
seuil_retenu = 0.5
y_pred_val_seuil = (y_proba_val >= seuil_retenu).astype(int)

print(f"Seuil retenu : {seuil_retenu}")
print(f"Recall (validation) a ce seuil : {recall_score(y_val, y_pred_val_seuil):.3f}")
print(f"Precision (validation) a ce seuil : {precision_score(y_val, y_pred_val_seuil):.3f}")

Seuil retenu : 0.5
Recall (validation) a ce seuil : 0.691
Precision (validation) a ce seuil : 0.227


---
# Étape 6 — L'unique évaluation finale, sur le test jamais consulté

**C'est la seule cellule de tout le pipeline qui regarde le jeu de test.** Toutes les décisions (quel modèle, quel C, quel seuil) ont été prises avant, uniquement avec train et validation. Ce qu'on obtient ici est donc une estimation honnête de la performance du modèle sur des données jamais vues, à aucun titre.

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

y_proba_test = modele_final.predict_proba(X_test_scaled)[:, 1]
y_pred_test = (y_proba_test >= seuil_retenu).astype(int)

auc_final = roc_auc_score(y_test, y_proba_test)
gini_final = 2 * auc_final - 1

from scipy.stats import ks_2samp
ks_final, _ = ks_2samp(y_proba_test[y_test == 1], y_proba_test[y_test == 0])

print("=== EVALUATION FINALE, SUR LE TEST (jamais consulte avant) ===")
print(f"AUC-ROC : {auc_final:.3f}")
print(f"Gini    : {gini_final:.3f}")
print(f"KS      : {ks_final:.3f}")
print()
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_test))
print()
print(classification_report(y_test, y_pred_test, target_names=["Non-defaut", "Defaut"]))

=== EVALUATION FINALE, SUR LE TEST (jamais consulte avant) ===
AUC-ROC : 0.762
Gini    : 0.523
KS      : 0.390

Matrice de confusion :
[[31113 14026]
 [ 1782  4149]]

              precision    recall  f1-score   support

  Non-defaut       0.95      0.69      0.80     45139
      Defaut       0.23      0.70      0.34      5931

    accuracy                           0.69     51070
   macro avg       0.59      0.69      0.57     51070
weighted avg       0.86      0.69      0.74     51070



## Comparer à ce qu'on avait annoncé en S8 (sur l'ancien test, potentiellement optimiste)

Si les chiffres sont proches, ça confirme que le biais introduit par la réutilisation du test en S6-S9 était limité en pratique sur ce dataset. S'ils diffèrent nettement, ça confirme au contraire que la correction était nécessaire.

In [9]:
print("S8 (evaluation potentiellement biaisee) : AUC = 0.762, Gini = 0.523, KS = 0.389")
print(f"Pipeline corrige (evaluation honnete)      : AUC = {auc_final:.3f}, Gini = {gini_final:.3f}, KS = {ks_final:.3f}")

S8 (evaluation potentiellement biaisee) : AUC = 0.762, Gini = 0.523, KS = 0.389
Pipeline corrige (evaluation honnete)      : AUC = 0.762, Gini = 0.523, KS = 0.390


## Étape 7 — Interprétabilité (sans risque, ne concerne pas le test)

In [10]:
coefficients = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": modele_final.coef_[0]
})
coefficients["Odds_ratio"] = np.exp(coefficients["Coefficient"])
coefficients["Magnitude"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Magnitude", ascending=False).drop(columns="Magnitude")

coefficients.head(5).round(3)

,Variable,Coefficient,Odds_ratio
0,Age,-0.603,0.547
26,LoanToIncomeRatio,0.470,1.600
6,InterestRate,0.465,1.591
4,MonthsEmployed,-0.343,0.709
11,HasCoSigner,-0.131,0.877


## Étape 8 — Sauvegarder le modèle final et les résultats

In [11]:
import joblib

joblib.dump(modele_final, "../src/models/modele_final_valide.joblib")
joblib.dump(scaler, "../src/models/scaler_final_valide.joblib")

resume = pd.DataFrame({
    "Metrique": ["AUC-ROC (test)", "Gini (test)", "KS (test)", "Seuil retenu", "Meilleur C"],
    "Valeur": [round(auc_final, 3), round(gini_final, 3), round(ks_final, 3), seuil_retenu, recherche.best_params_["C"]],
})
resume.to_csv("../reports/resume_final_valide.csv", index=False)
coefficients.to_csv("../reports/coefficients_finaux_valides.csv", index=False)

print("Modele, scaler et resultats sauvegardes.")

Modele, scaler et resultats sauvegardes.


## Résumé

- Découpage à trois (train 60% / validation 20% / test 20%), stratifié.
- Toutes les décisions itératives (choix du modèle, du C, du seuil) faites sur `validation`.
- **Une seule évaluation finale sur `test`**, jamais consultée avant — c'est le chiffre à citer comme performance réelle du projet.
- Comparaison avec les chiffres (potentiellement biaisés) de S6-S9, pour mesurer l'ampleur réelle du problème corrigé.

Ce pipeline remplace, pour toute évaluation finale citée dans le rapport (S10), les résultats précédemment obtenus en réutilisant le même jeu de test à travers S6-S9.